# WC-MNIST Parameter-Exploration — Pilot (3 arms) then Full Sweep

Runs the no-training parameter-exploration study: for each seed and config
(`simultaneous`, `sequential_noH`, `sequential_H`), a single **10-second
free-running simulation** of a freshly-initialised random Wilson-Cowan network
driven by MNIST — **no training** (no gradient steps, no epoch loop).

## Read this before running

The E→E connectivity was reworked (commits `09a0f60`, `a4aeb97`). Two things
changed that make **older bundles and figures incomparable** to new ones:

1. **Drive is now flat in `p_rewire`.** Both E→E scale ranges are `U[-0.6, 3.0]`
   and both coefficients are equal, so `p_rewire` is a pure topology knob.
   Previously drive fell 36.5% across the sampled `p` range with `corr = -1.000`,
   so every topology effect carried an unmodelled excitation difference.
2. **LFP electrodes moved** from stride 28 to stride 29. Stride 28 aliased with
   the image row length and put all 28 electrodes in image column 0, which
   carries no MNIST ink. **Existing hand-validated labels must be re-validated.**

Because absolute drive level rose, the pilot runs **three coefficient arms**
(`cfg.PILOT_COEFF_ARMS`) bracketing the Branch-B-matched value:

| arm | coefficient | per-unit E→E drive | vs Branch B (28.33) |
|---|---|---|---|
| `w075` | 0.075 | 24.04 | 0.849× |
| `w100` | 0.100 | 28.72 | 1.014× — matched |
| `w150` | 0.150 | 38.08 | 1.344× |

All three share **bit-identical initialisation** at a given seed (the
coefficients are consumed after every `init_rng` draw), so arm differences are
purely dynamical.

**Run the pilot (§6) first**, inspect it (§7), and only then raise `N_SEEDS` for
the full sweep. Do not skip to 500 seeds — if an arm saturates, the pilot is
where that becomes visible cheaply.

## Bundles and resume

Each run saves `arrays.npz` (E/I trajectories) + `params.json` + `manifest.json`,
plus figures, to Google Drive under `OUT_ROOT/arm/config/run_SSS/`. The sweep is
**resumable**: re-run after a disconnect and completed runs are skipped.

Resume also checks a **model fingerprint** — a hash of every constant that
changes what a run means. If the code changed since a bundle was written, that
bundle does *not* count as complete and is recomputed, so a mid-sweep edit can
never silently mix two model definitions in one output tree.

**Runtime**: ~3–4 s/run on GPU; measured **39 s/run on CPU** (6-core, 10 s
simulation), so a GPU runtime is effectively required. Pilot (3 arms × 3 configs
× 10 seeds = 90 runs) ≈ 6 min on GPU, ≈ 1 h on CPU. Full 3 × 3 × 500 = 4500 runs
≈ 5 h on GPU — consider one arm at a time, and check §7 between arms.
**Storage**: compressed f16 ≈ 5 MB/run → 90 runs ≈ 0.5 GB; 4500 ≈ 22 GB on Drive.
Confirm free Drive space before the full sweep.


## 1 · Mount Drive & choose output location

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# All bundles + master log land here. Safe to change; must be on Drive to persist.
# Layout: OUT_ROOT/<arm>/<config>/run_<seed>/{arrays.npz,params.json,manifest.json,figures/}
# The master log (param_explore_log.csv) sits at OUT_ROOT and spans every arm.
#
# NOTE: this is a NEW root, separate from any pre-rework sweep. The E->E rework
# changed both the dynamics and the LFP electrode positions, so old and new
# bundles are not comparable and must not share a tree. (Resume would catch it
# via the fingerprint, but keeping them apart is clearer.)
OUT_ROOT = '/content/drive/MyDrive/wc_mnist_param_explore_v3_scale0_5'
import os
os.makedirs(OUT_ROOT, exist_ok=True)
print('Output root:', OUT_ROOT)


## 2 · Clone the pinned code (private repo → needs a token)

The study repo is **private**, so Colab needs a GitHub token to clone it.
Do this once, securely — the token never appears in the notebook or on Drive:

1. Left sidebar → **🔑 Secrets**.
2. **+ Add new secret**, name it **`GH_TOKEN`**, paste a GitHub
   Personal Access Token with **`repo`** (read) scope as the value.
3. Toggle **Notebook access** ON for this notebook.

Create a token at github.com → Settings → Developer settings → Personal
access tokens. A fine-grained token scoped to just `AlexBoro/wc-mnist-study`
(Contents: Read) is enough.

*(Alternative: if you'd rather not manage a token, make the repo public and
this cell will clone anonymously.)*

The repo is pinned to a tagged commit so every run's `manifest.json` records
the exact code version.

In [ ]:
OWNER  = 'AlexBoro'
NAME   = 'wc-mnist-study'
TAG    = 'param-explore-v2.1'        # <-- pinned release tag (E->E rework + contact sheet)
CODE_DIR = '/content/wc-mnist-study'

import os, subprocess, sys

# Section 4 does os.chdir(CODE_DIR), so if the clone was deleted while this
# kernel was pointed inside it the cwd is now a dangling inode and EVERY
# subprocess dies with "fatal: Unable to read current working directory" --
# which is not an auth failure, though it used to be reported as one.
try:
    os.getcwd()
except (FileNotFoundError, OSError):
    os.chdir('/content')
    print('cwd was deleted (stale clone); moved to /content')

# Pull a token from Colab Secrets (falls back to anonymous for a public repo).
_token = None
try:
    from google.colab import userdata
    for _name in ('GH_TOKEN', 'GITHUB_TOKEN'):
        try:
            _token = userdata.get(_name)
            if _token:
                break
        except Exception:
            pass
except Exception:
    pass

_auth = f'x-access-token:{_token}@' if _token else ''
REPO = f'https://{_auth}github.com/{OWNER}/{NAME}.git'

# Clear a stale/partial dir from a previous failed clone.
if os.path.isdir(CODE_DIR) and not os.path.isdir(os.path.join(CODE_DIR, '.git')):
    import shutil; shutil.rmtree(CODE_DIR)

if not os.path.isdir(CODE_DIR):
    r = subprocess.run(['git','clone','--branch',TAG,'--depth','1',REPO,CODE_DIR],
                       capture_output=True, text=True)
    if r.returncode != 0:
        # Scrub any token from the error text before showing it.
        msg = r.stderr.replace(_token, '***') if _token else r.stderr
        # Match auth/visibility failures SPECIFICALLY.  A bare 'fatal:' test
        # catches every git error -- including a deleted cwd and a bad tag --
        # and misreports them as a missing token, sending you to Colab Secrets
        # to fix something that was never broken.
        _auth_fail = any(s in msg for s in (
            'Authentication failed', 'could not read Username',
            'terminal prompts disabled', 'Repository not found',
            'access rights', 'Permission denied'))
        _no_ref = 'Remote branch' in msg or 'not found in upstream' in msg
        if _auth_fail:
            raise RuntimeError(
                'Clone failed on AUTHENTICATION. The repo is private — add a '
                'GitHub token as a Colab Secret named GH_TOKEN (Contents:Read '
                'scope) and enable notebook access for it, then re-run.'
                f'\n\n{msg}')
        if _no_ref:
            raise RuntimeError(
                f'Clone reached the repo but tag/branch {TAG!r} does not exist '
                'on the remote. Check TAG above against the repo\'s tag list.'
                f'\n\n{msg}')
        raise RuntimeError(f'Clone failed (not an auth problem).\n\n{msg}')
    # Drop the token from the stored remote URL (belt-and-suspenders; /content is ephemeral).
    subprocess.run(['git','-C',CODE_DIR,'remote','set-url','origin',
                    f'https://github.com/{OWNER}/{NAME}.git'], check=False)
    print('cloned', NAME, 'at tag', TAG)
else:
    print('code dir exists; leaving as-is (delete /content/wc-mnist-study to re-clone)')

# Record the exact commit so manifests carry it even without a .git in the run cwd.
commit = subprocess.run(['git','-C',CODE_DIR,'rev-parse','HEAD'],
                        capture_output=True, text=True).stdout.strip()
os.environ['WC_CODE_COMMIT'] = commit
print('Pinned commit:', commit)

if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)


## 3 · Dependencies

Colab ships torch + torchvision + numpy + scipy. Install only what's missing.

In [ ]:
import importlib, subprocess, sys
for pkg in ['torch','torchvision','numpy','scipy','matplotlib']:
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.run([sys.executable,'-m','pip','install','-q',pkg], check=True)

import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available(),
      '|', (torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'))


## 4 · Sweep configuration

**Pilot first.** `N_SEEDS = 10` with all three arms is the sanity check; only
raise it after inspecting §7. `ARMS = None` runs a single unarmed sweep with the
study-default coefficients (`cfg.W_CELOC`), reproducing the original layout.


In [ ]:
import numpy as np
os.chdir(CODE_DIR)                       # so wc_experiment resolves MNIST + git

import wc_config as cfg
import param_explore as PE
import param_persist as PP
import param_render as PR

# --- knobs -------------------------------------------------------------------
N_SEEDS       = 20                       # pilot; raise only after the four checks in s7 pass
#   E->E init scale is now U(-0.6, 3.0) for BOTH CeLoc and CeRem (wc_config), and
#   feature extraction drops the first cfg.LEAD_IN_S = 1.0 s -- see wc_config for why.
SEEDS         = list(range(N_SEEDS))
CONFIG_NAMES  = [c.name for c in cfg.make_configs()]   # simultaneous, sequential_noH, sequential_H
ARMS          = cfg.PILOT_COEFF_ARMS     # 3 coefficient arms; None = single unarmed sweep
RUN_SECONDS   = 10.0
RECORD_STRIDE = 1                        # 1 = full dt resolution (5000 steps); 2 halves storage
DTYPE_OUT     = np.float16               # f16 compressed ~5 MB/run
RENDER_FIGS   = True                     # write heatmap_E.png + lfp.png per run
DEEP_CHECK    = False                    # True verifies E_traj rows on skip (opens npz: slow on Drive)
# -----------------------------------------------------------------------------

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
render_fn = PR.render_from_result if RENDER_FIGS else None

n_arms = len(ARMS) if ARMS else 1
total  = n_arms * len(CONFIG_NAMES) * len(SEEDS)
print(f'{n_arms} arms x {len(CONFIG_NAMES)} configs x {len(SEEDS)} seeds = {total} runs')
print(f'device={device} | stride={RECORD_STRIDE} | dtype={np.dtype(DTYPE_OUT).name}')
print(f'est. runtime {total*3.5/60:.0f}-{total*4.5/60:.0f} min | est. storage {total*5/1024:.1f} GB')

# The fingerprint of what we are ABOUT to run. Anything on disk under a different
# fingerprint will be recomputed, not reused.
print()
for arm, w in (ARMS or {'': None}).items():
    import dataclasses
    c = cfg.make_config(CONFIG_NAMES[0])
    if w is not None:
        c = dataclasses.replace(c, w_celoc=float(w), w_cerem=float(w))
    fp = PE.model_fingerprint(c)
    drive = cfg.W_PY2PY_SELF + cfg.TARGET_OUTDEG_PyPy * fp['w_celoc'] * 1.2
    print(f"  arm {arm or '(default)':10s} w={fp['w_celoc']:.3f}  "
          f"E->E drive {drive:5.2f}  fingerprint {fp['hash']}")
print(f"\n  LFP_SPAN = {cfg.LFP_SPAN}  (provisional -- recalibrate in section 7)")


## 5 · MNIST download (once)

Downloads MNIST into the repo's data root; subsequent runs reuse it.

In [ ]:
from torchvision import datasets, transforms
import wc_experiment as X
data_root = X._resolve_data_root()
datasets.MNIST(root=data_root, download=True, train=True, transform=transforms.ToTensor())
print('MNIST ready at', data_root)


## 6 · Run the sweep (resumable)

Safe to interrupt and re-run: completed bundles are skipped, and a disconnect
costs at most the in-flight run. A run that raises is logged to `failures.jsonl`
and the sweep continues — it does not abandon the remaining runs.

Bundles whose fingerprint does not match the current code are reported as
`refreshed` and recomputed.


In [ ]:
result = PP.run_sweep(
    OUT_ROOT, seeds=SEEDS, config_names=CONFIG_NAMES, device=device,
    run_seconds=RUN_SECONDS, record_stride=RECORD_STRIDE, dtype_out=DTYPE_OUT,
    code_tag=TAG, repo_dir=CODE_DIR, render_fn=render_fn, verbose=True,
    arms=ARMS, deep_check=DEEP_CHECK)
print(result)
if result['failed']:
    print(f"\n{result['failed']} run(s) FAILED -- see {OUT_ROOT}/failures.jsonl")


## 7 · Inspect the pilot

Four checks, in the order that matters. **Do not raise `N_SEEDS` until all four
pass.**

1. **Completeness** — every (arm, config, seed) present under the current fingerprint.
2. **LFP span** — is `cfg.LFP_SPAN` still right after the drive change? The fixed
   y-axis is what makes a pixel-deflection threshold transferable, so clipping or
   excessive slack both matter.
3. **Regime occupancy** — the mean-field estimate said the higher arms push
   initialisation toward saturation. This measures it on the real dynamics, which
   the mean-field figure could not (it ignored in-degree heterogeneity, the
   negative-weight fraction, and the dynamics).
4. **Flat drive held** — `corr(p_rewire, drive)` should be ~0 in every arm. This is
   the property the whole rework exists for; if it is not ~0, stop.


In [ ]:
# ---- 1. completeness -------------------------------------------------------
import os, csv, json, glob
status = PP.sweep_status(OUT_ROOT, SEEDS, CONFIG_NAMES, arms=ARMS)
print(f"{'arm':10s} {'config':16s} {'complete':>9} {'missing':>8} {'stale':>6}")
for r in status:
    flag = '' if r['missing'] == 0 and r['stale'] == 0 else '   <-- rerun section 6'
    print(f"{r['arm'] or '-':10s} {r['config_name']:16s} {r['complete']:9d} "
          f"{r['missing']:8d} {r['stale']:6d}{flag}")
print(f"\nTOTAL complete {sum(r['complete'] for r in status)}/{sum(r['total'] for r in status)}"
      f" | stale {sum(r['stale'] for r in status)} (written under a different model definition)")

fail_path = os.path.join(OUT_ROOT, 'failures.jsonl')
if os.path.exists(fail_path):
    fails = [json.loads(l) for l in open(fail_path)]
    print(f"\n{len(fails)} recorded failure(s); most recent:")
    for f in fails[-3:]:
        print(f"  arm={f['arm'] or '-'} {f['config_name']} seed={f['seed']}: {f['error']}")

log_path = os.path.join(OUT_ROOT, PP.MASTER_LOG)
rows = list(csv.DictReader(open(log_path))) if os.path.exists(log_path) else []
print(f"\nmaster log: {len(rows)} rows, {len(rows[0]) if rows else 0} columns")


In [ ]:
# ---- 2. LFP span calibration ----------------------------------------------
# The LFP y-axis is deliberately NOT autoscaled: a constant span makes
# pixels-per-signal-unit identical on every figure, which is what lets a fixed
# pixel-deflection threshold transfer between runs. cfg.LFP_SPAN = 150 was
# calibrated against the OLD coefficients, so re-check it here.
#
# The span must cover the electrode LADDER ((28-1)*5 = 135) plus the signal
# excursion above and below it. Headroom = LFP_SPAN - 135.
import dataclasses
LADDER = (28 - 1) * 5.0
print(f"cfg.LFP_SPAN = {cfg.LFP_SPAN}   ladder = {LADDER}   headroom = {cfg.LFP_SPAN - LADDER}\n")
print(f"{'arm':8s} {'config':16s} {'seed':>4} {'peak|LFP|':>10} {'span_needed':>12} {'clipped':>8}")
worst = {}
for arm in (ARMS or {'': None}):
    for cname in CONFIG_NAMES:
        for seed in SEEDS[:3]:                     # 3 seeds/cell is enough to size the axis
            try:
                st = PR.render_from_bundle(OUT_ROOT, cname, seed, arm=arm, return_stats=True)
            except Exception as e:
                print(f"{arm or '-':8s} {cname:16s} {seed:4d}   (skipped: {type(e).__name__})")
                continue
            print(f"{arm or '-':8s} {cname:16s} {seed:4d} {st['peak_abs']:10.2f} "
                  f"{st['span_needed']:12.1f} {100*st['clipped_frac']:7.2f}%")
            worst[arm] = max(worst.get(arm, 0.0), st['span_needed'])
print()
for arm, need in worst.items():
    verdict = ('OK' if need <= cfg.LFP_SPAN else
               f'CLIPPING -- raise cfg.LFP_SPAN to >= {int(np.ceil(need/10)*10)}')
    print(f"  arm {arm or '(default)':10s} max span_needed {need:7.1f}  ->  {verdict}")
print("\nIf you change cfg.LFP_SPAN, re-render (not re-simulate): the arrays are")
print("unaffected, so loop render_from_bundle over the tree. Changing it does NOT")
print("change the fingerprint -- it is a figure constant, not a model constant.")


In [ ]:
# ---- 3. regime occupancy on the real dynamics ------------------------------
# The mean-field estimate predicted the higher arms sit closer to saturation.
# That estimate ignored in-degree heterogeneity, the negative-weight fraction,
# and the dynamics -- so measure the actual E trajectories.
QUIET, SATUR = 0.05, 0.95
print(f"{'arm':8s} {'config':16s} {'mean E':>8} {'sd E':>7} "
      f"{'%quiet':>8} {'%satur':>8} {'%runs pinned':>13}")
for arm in (ARMS or {'': None}):
    for cname in CONFIG_NAMES:
        means, quiets, saturs, pinned, n = [], [], [], 0, 0
        for seed in SEEDS:
            try:
                arrays, _p, _m = PP.load_bundle(OUT_ROOT, cname, seed, arm=arm)
            except Exception:
                continue
            E = arrays['E_traj'].astype(np.float32)
            half = E[E.shape[0] // 2:]                 # 2nd half: past initial transient
            means.append(half.mean())
            quiets.append((half < QUIET).mean())
            saturs.append((half > SATUR).mean())
            pinned += int(half.mean() > SATUR); n += 1
        if not n:
            continue
        print(f"{arm or '-':8s} {cname:16s} {np.mean(means):8.3f} {np.std(means):7.3f} "
              f"{100*np.mean(quiets):7.1f}% {100*np.mean(saturs):7.1f}% "
              f"{100*pinned/n:12.1f}%")
print("\n'%runs pinned' is the number that decides the arm: a pinned network cannot")
print("show a seizure transition because it is already saturated. If an arm is mostly")
print("pinned, it is not a usable operating point regardless of its drive figure.")


In [ ]:
# ---- 4. flat drive held (the property the rework exists for) ---------------
# corr(p_rewire, E->E drive) alone is a MISLEADING test here, and the pilot will
# show you why: with a tiny-but-systematic residual, correlation approaches 1
# while the effect size is negligible. Correlation measures whether the residual
# points consistently in one direction; it says nothing about how big it is.
#
# The substantive criterion is the SWING as a fraction of drive. The old
# construction swung 8.77 units = 36.5% of drive. Anything under ~1% is
# irrelevant to the dynamics no matter what the correlation says.
print(f"{'arm':8s} {'n':>4} {'mean drive':>11} {'swing':>8} {'% of drive':>11} {'corr':>8}  verdict")
for arm, w in (ARMS or {'': None}).items():
    ps, ds = [], []
    for cname in CONFIG_NAMES:
        for seed in SEEDS:
            try:
                _a, params, _m = PP.load_bundle(OUT_ROOT, cname, seed, arm=arm)
            except Exception:
                continue
            nL, nR = params['n_CeLoc_edges'], params['n_CeRem_edges']
            mL, mR = params['CeLoc_values_mean'], params['CeRem_values_mean']
            wl = params.get('w_celoc', cfg.W_CELOC); wr = params.get('w_cerem', cfg.W_CEREM)
            N = params['N']
            ds.append(cfg.W_PY2PY_SELF + (wl * nL * mL + wr * nR * mR) / N)
            ps.append(nR / max(nL + nR, 1))     # realised remote fraction = realised p_rewire
    if len(ps) < 4:
        print(f"{arm or '-':8s} {len(ps):4d}   (too few bundles)"); continue
    ps, ds = np.array(ps), np.array(ds)
    slope = np.polyfit(ps, ds, 1)[0]
    corr = np.corrcoef(ps, ds)[0, 1]
    swing = abs(slope) * (ps.max() - ps.min())
    pct = 100 * swing / ds.mean()
    if pct < 1.0:
        verdict = 'FLAT'
    elif pct < 5.0:
        verdict = f'small residual ({pct:.1f}%) -- acceptable, note it'
    else:
        verdict = f'NOT FLAT ({pct:.1f}% of drive) -- STOP, investigate'
    print(f"{arm or '-':8s} {len(ps):4d} {ds.mean():11.2f} {swing:8.3f} {pct:10.2f}% "
          f"{corr:+8.3f}  {verdict}")
print()
print("Reading the corr column: it is expected to be IDENTICAL across arms (drive is")
print("an affine function of the same connectivity in each, so the correlation cannot")
print("differ) and it is expected to be noisy and often large at pilot n -- |corr| ~ 0.8")
print("at n = 5 is not significant. Measured over 400 seeds pre-flight the slope was")
print("+0.0036 +- 0.0172 (95% CI), i.e. not distinguishable from zero. Judge by the")
print("'% of drive' column; use corr only to check the residual has no large magnitude")
print("AND no consistent sign once n is in the hundreds.")


## 8 · Contact sheet — scroll and compare every figure

`build_contact_sheet.py` writes ONE self-contained `.html` holding thumbnails of
every run, so the whole sweep is scrollable in a browser.

**Row = (config, seed); column = coefficient arm.** The three arms at a given
seed+config share bit-identical initialization and differ *only* in E→E drive,
so reading **left→right along a row** isolates the coefficient's effect with
everything else held fixed. Reading **down a column** shows seed-to-seed spread
within one arm. That pairing is the comparison the pilot exists to make.

Controls in the sheet: filter by config, show LFP only / heatmap only / both,
sort by saturation or mean E (worst arm first) to bring problem runs to the top,
and click any panel for a full-window view.

> **Thumbnails are downscaled JPEGs — never measure them.** Detector thresholds
> must be set from the full-resolution PNGs, which every card links to. This is
> the same trap as `detector_validation_seeds0-4.html`.


In [ ]:
# Build the contact sheet over everything the sweep has written so far.
# Safe to re-run: it re-reads the tree each time and overwrites the html.
import importlib, sys, os
sys.path.insert(0, CODE_DIR) if CODE_DIR not in sys.path else None
import build_contact_sheet as BCS
importlib.reload(BCS)

# DYNAMICS=True adds mean_E / saturation-occupancy per card and enables the
# sort-by-saturation control.  It reads E_traj from every arrays.npz (~31 MB per
# run), so the first build is slow; results cache to contact_sheet_dynamics.csv
# at the sweep root and later builds reuse them.
DYNAMICS = True

sheet = BCS.main([
    '--out-root', OUT_ROOT,
    '--html', 'contact_sheet.html',
    '--thumb-width', '460',
] + (['--dynamics'] if DYNAMICS else []))

print(f"\nsheet: {sheet}  ({os.path.getsize(sheet)/1e6:.1f} MB)")
print("It is on Drive -- but Drive will NOT render html in the browser.")
print("Use one of the two cells below to actually look at it.")


In [ ]:
# --- how to view it -------------------------------------------------------
# (a) DOWNLOAD and open locally -- best for real inspection: full window, fast
#     scrolling, and the full-res links work if you also have the tree synced.
from google.colab import files
files.download(str(sheet))

# (b) INLINE preview, a few seeds only -- convenient but cramped, and a big
#     file makes the notebook sluggish, hence the --max-seed cap.
# from IPython.display import HTML, display
# preview = BCS.main(['--out-root', OUT_ROOT, '--html', 'contact_sheet_preview.html',
#                     '--max-seed', '1', '--thumb-width', '380'])
# display(HTML(open(preview).read()))

# (c) One config at a time, when the full sheet gets unwieldy on the 500-seed run:
# BCS.main(['--out-root', OUT_ROOT, '--config', 'sequential_H',
#           '--html', 'contact_sheet_sequential_H.html', '--dynamics'])


## 9 · (optional) Export for local analysis

The master log is the input to the classification step; the full bundles are only
needed for re-rendering or re-measuring.


In [ ]:
# Zip the pilot's figures (small) for offline inspection, or a few full bundles.
# import shutil, glob
# stage = '/content/pe_figs'; os.makedirs(stage, exist_ok=True)
# for arm in (ARMS or {'': None}):
#     for cname in CONFIG_NAMES:
#         for seed in SEEDS[:3]:
#             src = PP.run_dir(OUT_ROOT, cname, seed, arm=arm) / 'figures'
#             if src.exists():
#                 dst = os.path.join(stage, f'{arm or "default"}__{cname}__run{seed:03d}')
#                 shutil.copytree(src, dst, dirs_exist_ok=True)
# shutil.make_archive('/content/pe_figs', 'zip', stage)
# from google.colab import files; files.download('/content/pe_figs.zip')

# The master log alone is usually what you want next (it drives classification):
# from google.colab import files
# files.download(os.path.join(OUT_ROOT, PP.MASTER_LOG))
